In [2]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [11]:
with open(r"D:\Test\Python\Deep Learning\micrograd\files\names.txt","r") as f:
    words = f.read().splitlines()
words[:5]

['emma', 'olivia', 'ava', 'isabella', 'sophia']

In [21]:
chars = sorted(list(set("".join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
vocab_size = len(itos)
vocab_size

27

In [38]:
block_size = 8

def build_dataset(words):
    X,Y = [],[]

    for w in words:
        context = [0] * block_size

        for ch in w + ".":
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]

    X = torch.tensor(X)
    Y = torch.tensor(Y)
    print(X.shape,Y.shape)
    return X,Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr,Ytr = build_dataset(words[:n1])
Xdev,Ydev = build_dataset(words[n1:n2])
Xte,Yte = build_dataset(words[n2:])

torch.Size([182580, 8]) torch.Size([182580])
torch.Size([22767, 8]) torch.Size([22767])
torch.Size([22799, 8]) torch.Size([22799])


In [47]:
for x,y in zip(Xtr[:20],Ytr[:20]):
    print("".join(itos[i.item()] for i in x), "-->",itos[y.item()])

........ --> e
.......e --> b
......eb --> r
.....ebr --> i
....ebri --> m
...ebrim --> a
..ebrima --> .
........ --> h
.......h --> i
......hi --> l
.....hil --> t
....hilt --> o
...hilto --> n
..hilton --> .
........ --> j
.......j --> h
......jh --> e
.....jhe --> n
....jhen --> e
...jhene --> .


In [ ]:
class Linear:
    def __init__(self,fan_in,fan_out,bias=True):
        self.weights = torch.randn((fan_in,fan_out)) / fan_in ** 0.5
        self.bias = torch.zeros(fan_out) if bias else None

    def __call__(self,x):
        self.out = x @ self.weights
        if self.bias:
            self.out += self.bias
        return self.out

    def parameters(self):
        return [self.weights] + ([] if self.bias is None else [self.bias])

class BatchNorm1D:
    def __init__(self,dim,eps=1e-5,momentum=0.1):
        self.eps = eps
        self.momentum = momentum
        self.training = True

        self.gamma = torch.ones(dim)
        self.beta = torch.zeros(dim)

        self.running_mean = torch.zeros(dim)
        self.running_var = torch.ones(dim)

    def __call__(self,x):
        if self.training:
            xmean = x.mean(0,True)
            xvar = x.var(0,True)
        else:
            xmean = self.running_mean
            xvar = self.running_var

        xhat = (x-xmean)/torch.sqrt(xvar + self.eps)
        self.out = self.gamma * xhat + self.beta

        if self.training:
                    with torch.no_grad():
                        # Update the bufferes using Exponential Moving Average (EMA) with momentum
                        self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * xmean
                        self.running_var = (1 - self.momentum) * self.running_var + self.momentum * xvar
        return self.out
        
    def parameters(self):
        return [self.gamma,self.beta]

class Tanh:
    def __call__(self,x):
          self.out = torch.tanh(x)
          return self.out

    def parameters(self):
         return []


class Embeddings:
    def __init__(self,n_embd,embd_dim):
          self.weights = torch.randn((n_embd,embd_dim))

    def __call__(self,x):
         self.out = self.weights[x]
         return self.out

    def parameters(self):
        return [self.weights]

class Flatten:
    def __call__(self,x):
         self.out = x.view(x.shape[0],-1)    

    def parameters(self):
         return []
